In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray


# ============================================================
# CONFIGURATION
# ============================================================

AVG_FILE = Path("data_stream-oper_stepType-avg.nc")
INSTANT_FILE = Path("data_stream-oper_stepType-instant.nc")
MAX_FILE = Path("data_stream-oper_stepType-max.nc")
PTYPE_FILE = Path("ptype_hopefully.nc")

OUTPUT_DIR = Path("derived/era5_icing")
RASTER_DIR = OUTPUT_DIR / "rasters"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RASTER_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# PRECIPITATION TYPE CODES
# WMO / ECMWF GRIB2 Code Table 4.201
# ============================================================

PTYPE_LABELS = {
    0: "No precipitation",
    1: "Rain",
    2: "Thunderstorm",
    3: "Freezing rain",
    4: "Mixed / ice",
    5: "Snow",
    6: "Wet snow",
    7: "Mixture of rain and snow",
    8: "Ice pellets",
    9: "Graupel",
    10: "Hail",
    11: "Drizzle",
    12: "Freezing drizzle",
    13: "Hail < 5 mm",
    14: "Hail >= 5 mm",
    255: "Missing",
}


# ============================================================
# HELPERS
# ============================================================

def normalize_dataset(ds):
    """
    Standardize coordinate names, remove irrelevant auxiliary
    coordinates, and convert longitude to -180..180.
    """

    rename = {}

    if "time" in ds.dims and "valid_time" not in ds.dims:
        rename["time"] = "valid_time"

    if "lat" in ds.dims and "latitude" not in ds.dims:
        rename["lat"] = "latitude"

    if "lon" in ds.dims and "longitude" not in ds.dims:
        rename["lon"] = "longitude"

    if rename:
        ds = ds.rename(rename)

    # Convert 0..360 longitude convention to -180..180.
    if float(ds.longitude.max()) > 180:
        ds = ds.assign_coords(
            longitude=((ds.longitude + 180) % 360) - 180
        ).sortby("longitude")

    # Drop auxiliary non-index coordinates such as expver/number.
    for coord in list(ds.coords):
        if coord not in {"valid_time", "latitude", "longitude"}:
            ds = ds.drop_vars(coord)

    return ds


def align_to_reference(ds, reference):
    """
    Confirm that ds represents the same grid/time axis as
    reference and then adopt reference's exact coordinates.

    This avoids tiny floating-point coordinate mismatches.
    """

    assert ds.sizes["valid_time"] == reference.sizes["valid_time"]
    assert ds.sizes["latitude"] == reference.sizes["latitude"]
    assert ds.sizes["longitude"] == reference.sizes["longitude"]

    assert np.array_equal(
        ds.valid_time.values,
        reference.valid_time.values,
    )

    assert np.allclose(
        ds.latitude.values,
        reference.latitude.values,
    )

    assert np.allclose(
        ds.longitude.values,
        reference.longitude.values,
    )

    return ds.assign_coords(
        valid_time=reference.valid_time,
        latitude=reference.latitude,
        longitude=reference.longitude,
    )


def export_geotiff(data_array, filename):
    """
    Export a 2-D derived field as an EPSG:4326 GeoTIFF.
    QGIS can reproject it on the fly to EPSG:5070.
    """

    da = data_array.copy()

    if "quantile" in da.coords:
        da = da.drop_vars("quantile")

    da = da.rio.set_spatial_dims(
        x_dim="longitude",
        y_dim="latitude",
    )

    da = da.rio.write_crs("EPSG:4326")

    output_path = RASTER_DIR / filename

    da.rio.to_raster(
        output_path,
        compress="DEFLATE",
    )

    print(f"Saved: {output_path}")


def percentage_of_hours(mask):
    """
    Percentage of valid hourly timesteps satisfying mask.
    """
    return mask.mean(dim="valid_time") * 100


def percentage_of_precip_hours(event_mask, precip_mask):
    """
    Of all hours classified as precipitation, calculate the
    percentage belonging to event_mask.
    """

    numerator = event_mask.sum(dim="valid_time")
    denominator = precip_mask.sum(dim="valid_time")

    return xr.where(
        denominator > 0,
        numerator / denominator * 100,
        np.nan,
    )


# ============================================================
# 1. LOAD ALL FOUR SOURCE FILES
# ============================================================

avg = normalize_dataset(
    xr.open_dataset(AVG_FILE)
)

instant = normalize_dataset(
    xr.open_dataset(INSTANT_FILE)
)

maximum = normalize_dataset(
    xr.open_dataset(MAX_FILE)
)

ptype_ds = normalize_dataset(
    xr.open_dataset(PTYPE_FILE)
)


print("AVG variables:")
print(list(avg.data_vars))

print("\nINSTANT variables:")
print(list(instant.data_vars))

print("\nMAX variables:")
print(list(maximum.data_vars))

print("\nPTYPE variables:")
print(list(ptype_ds.data_vars))


# ============================================================
# 2. ALIGN EVERYTHING TO THE INSTANTANEOUS ERA5 GRID
# ============================================================

avg = align_to_reference(
    avg,
    instant,
)

maximum = align_to_reference(
    maximum,
    instant,
)

ptype_ds = align_to_reference(
    ptype_ds,
    instant,
)


era5 = xr.merge(
    [
        instant,
        avg,
        maximum,
        ptype_ds,
    ],
    join="exact",
    compat="no_conflicts",
)


assert era5.sizes["valid_time"] == 744
assert era5.sizes["latitude"] == 21
assert era5.sizes["longitude"] == 29


print("\nMerged dimensions:")
print(era5.sizes)

print("\nMerged variables:")
print(list(era5.data_vars))


# ============================================================
# 3. UNIT CONVERSIONS / BASIC DERIVED VARIABLES
# ============================================================

# ------------------------------------------------------------
# Air temperature
# ------------------------------------------------------------

era5["t2m_c"] = (
    era5["t2m"] - 273.15
)

era5["t2m_c"].attrs = {
    "long_name": "2 metre air temperature",
    "units": "degC",
}


# ------------------------------------------------------------
# Dewpoint
# ------------------------------------------------------------

era5["d2m_c"] = (
    era5["d2m"] - 273.15
)

era5["d2m_c"].attrs = {
    "long_name": "2 metre dewpoint temperature",
    "units": "degC",
}


# ------------------------------------------------------------
# Dewpoint depression
#
# T - Td
#
# Small values indicate air close to saturation.
# This is climate/icing context, NOT an icing criterion.
# ------------------------------------------------------------

era5["dewpoint_depression_c"] = (
    era5["t2m_c"]
    - era5["d2m_c"]
)

era5["dewpoint_depression_c"].attrs = {
    "long_name": "2 metre dewpoint depression",
    "units": "degC",
}


# ------------------------------------------------------------
# Mean precipitation rate
#
# kg m^-2 s^-1 -> mm/hour
#
# 1 kg/m² liquid water = 1 mm water depth.
# ------------------------------------------------------------

era5["precip_rate_mmh"] = (
    era5["avg_tprate"] * 3600
)

era5["precip_rate_mmh"].attrs = {
    "long_name": "Mean total precipitation rate",
    "units": "mm h-1",
}


# ------------------------------------------------------------
# Mean snowfall rate
#
# This remains WATER EQUIVALENT, not physical snow depth.
# ------------------------------------------------------------

era5["snowfall_rate_mmh"] = (
    era5["avg_tsrwe"] * 3600
)

era5["snowfall_rate_mmh"].attrs = {
    "long_name": "Mean snowfall water-equivalent rate",
    "units": "mm h-1 water equivalent",
}


# ------------------------------------------------------------
# Low cloud cover
# ------------------------------------------------------------

if float(era5["lcc"].max()) <= 1.5:
    era5["low_cloud_cover_pct"] = (
        era5["lcc"] * 100
    )
else:
    era5["low_cloud_cover_pct"] = (
        era5["lcc"]
    )

era5["low_cloud_cover_pct"].attrs = {
    "long_name": "Low cloud cover",
    "units": "%",
}


# ============================================================
# 4. GENERAL TEMPERATURE / HUMIDITY CONTEXT
# ============================================================

mean_temp = (
    era5["t2m_c"]
    .mean(dim="valid_time")
)

mean_temp.name = "mean_temperature"
mean_temp.attrs = {
    "long_name": "Mean January air temperature",
    "units": "degC",
}


freezing_pct = (
    (era5["t2m_c"] < 0)
    .mean(dim="valid_time")
    * 100
)

freezing_pct.name = "freezing_hours_pct"
freezing_pct.attrs = {
    "long_name": (
        "Percentage of January hours "
        "with 2 metre temperature below 0 degC"
    ),
    "units": "%",
}


mean_dewpoint_depression = (
    era5["dewpoint_depression_c"]
    .mean(dim="valid_time")
)

mean_dewpoint_depression.name = (
    "mean_dewpoint_depression"
)

mean_dewpoint_depression.attrs = {
    "long_name": "Mean January dewpoint depression",
    "units": "degC",
}


# Exploratory saturation proxy.
near_saturation_pct = (
    (era5["dewpoint_depression_c"] <= 2)
    .mean(dim="valid_time")
    * 100
)

near_saturation_pct.name = (
    "near_saturation_hours_pct"
)

near_saturation_pct.attrs = {
    "long_name": (
        "Percentage of January hours "
        "with dewpoint depression <= 2 degC"
    ),
    "units": "%",
}


# ============================================================
# 5. CLOUD CONTEXT
# ============================================================

mean_low_cloud = (
    era5["low_cloud_cover_pct"]
    .mean(dim="valid_time")
)

mean_low_cloud.name = "mean_low_cloud_cover"

mean_low_cloud.attrs = {
    "long_name": "Mean January low cloud cover",
    "units": "%",
}


low_cloud_50_pct = (
    (era5["low_cloud_cover_pct"] >= 50)
    .mean(dim="valid_time")
    * 100
)

low_cloud_50_pct.name = (
    "low_cloud_ge50_hours_pct"
)

low_cloud_50_pct.attrs = {
    "long_name": (
        "Percentage of January hours "
        "with at least 50 percent low cloud cover"
    ),
    "units": "%",
}


median_cloud_base = (
    era5["cbh"]
    .median(
        dim="valid_time",
        skipna=True,
    )
)

median_cloud_base.name = (
    "median_cloud_base_height"
)

median_cloud_base.attrs = {
    "long_name": "Median January cloud base height",
    "units": era5["cbh"].attrs.get(
        "units",
        "m",
    ),
}


# ============================================================
# 6. SUPERCOOLED LIQUID WATER
# ============================================================

mean_tcslw = (
    era5["tcslw"]
    .mean(dim="valid_time")
)

mean_tcslw.name = "mean_tcslw"

mean_tcslw.attrs = {
    "long_name": (
        "Mean total-column supercooled liquid water"
    ),
    "units": era5["tcslw"].attrs.get(
        "units",
        "kg m-2",
    ),
}


max_tcslw = (
    era5["tcslw"]
    .max(dim="valid_time")
)

max_tcslw.name = "max_tcslw"

max_tcslw.attrs = {
    "long_name": (
        "Maximum total-column supercooled liquid water"
    ),
    "units": era5["tcslw"].attrs.get(
        "units",
        "kg m-2",
    ),
}


# Restrict to subfreezing surface conditions.
# This is still NOT a conductor ice-load calculation.
mean_tcslw_subfreezing = (
    era5["tcslw"]
    .where(
        era5["t2m_c"] < 0
    )
    .mean(
        dim="valid_time",
        skipna=True,
    )
)

mean_tcslw_subfreezing.name = (
    "mean_tcslw_subfreezing"
)

mean_tcslw_subfreezing.attrs = {
    "long_name": (
        "Mean total-column supercooled liquid water "
        "during subfreezing surface conditions"
    ),
    "units": era5["tcslw"].attrs.get(
        "units",
        "kg m-2",
    ),
}


# ============================================================
# 7. WIND GUST METRICS
# ============================================================

max_instant_gust = (
    era5["i10fg"]
    .max(dim="valid_time")
)

max_instant_gust.name = (
    "max_instantaneous_gust"
)

max_instant_gust.attrs = {
    "long_name": (
        "Maximum instantaneous 10 metre wind gust"
    ),
    "units": "m s-1",
}


p95_instant_gust = (
    era5["i10fg"]
    .quantile(
        0.95,
        dim="valid_time",
    )
)

if "quantile" in p95_instant_gust.coords:
    p95_instant_gust = (
        p95_instant_gust.drop_vars("quantile")
    )

p95_instant_gust.name = (
    "p95_instantaneous_gust"
)

p95_instant_gust.attrs = {
    "long_name": (
        "95th percentile instantaneous "
        "10 metre wind gust"
    ),
    "units": "m s-1",
}


# fg10 represents the maximum gust over the preceding
# post-processing interval.
max_interval_gust = (
    era5["fg10"]
    .max(dim="valid_time")
)

max_interval_gust.name = "max_interval_gust"

max_interval_gust.attrs = {
    "long_name": (
        "Maximum 10 metre interval wind gust"
    ),
    "units": "m s-1",
}


p95_interval_gust = (
    era5["fg10"]
    .quantile(
        0.95,
        dim="valid_time",
    )
)

if "quantile" in p95_interval_gust.coords:
    p95_interval_gust = (
        p95_interval_gust.drop_vars("quantile")
    )

p95_interval_gust.name = "p95_interval_gust"

p95_interval_gust.attrs = {
    "long_name": (
        "95th percentile 10 metre interval wind gust"
    ),
    "units": "m s-1",
}


# ============================================================
# 8. PRECIPITATION / SNOWFALL RATE METRICS
# ============================================================

mean_precip_rate = (
    era5["precip_rate_mmh"]
    .mean(dim="valid_time")
)

mean_precip_rate.name = "mean_precip_rate"

mean_precip_rate.attrs = {
    "long_name": "Mean January precipitation rate",
    "units": "mm h-1",
}


max_precip_rate = (
    era5["precip_rate_mmh"]
    .max(dim="valid_time")
)

max_precip_rate.name = "max_precip_rate"

max_precip_rate.attrs = {
    "long_name": "Maximum January precipitation rate",
    "units": "mm h-1",
}


mean_snowfall_rate = (
    era5["snowfall_rate_mmh"]
    .mean(dim="valid_time")
)

mean_snowfall_rate.name = "mean_snowfall_rate"

mean_snowfall_rate.attrs = {
    "long_name": (
        "Mean January snowfall water-equivalent rate"
    ),
    "units": "mm h-1 water equivalent",
}


max_snowfall_rate = (
    era5["snowfall_rate_mmh"]
    .max(dim="valid_time")
)

max_snowfall_rate.name = "max_snowfall_rate"

max_snowfall_rate.attrs = {
    "long_name": (
        "Maximum January snowfall water-equivalent rate"
    ),
    "units": "mm h-1 water equivalent",
}


# ============================================================
# 9. PRECIPITATION-TYPE MASKS
# ============================================================

ptype = era5["ptype"]

valid_ptype = (
    np.isfinite(ptype)
    & (ptype != 255)
)

precipitation_occurring = (
    valid_ptype
    & (ptype != 0)
)

freezing_rain = (
    valid_ptype
    & (ptype == 3)
)

wet_snow = (
    valid_ptype
    & (ptype == 6)
)

mixed_rain_snow = (
    valid_ptype
    & (ptype == 7)
)

ice_pellets = (
    valid_ptype
    & (ptype == 8)
)

freezing_drizzle = (
    valid_ptype
    & (ptype == 12)
)


# Supercooled liquid precipitation:
# directly relevant to glaze formation.
freezing_liquid_precip = (
    freezing_rain
    | freezing_drizzle
)


# Broader conductor-accretion-relevant precipitation:
# glaze mechanisms + adhesive wet snow.
#
# This is NOT called "icing hours", because ERA5 does not
# calculate conductor ice accretion.
accretion_relevant_precip = (
    freezing_rain
    | freezing_drizzle
    | wet_snow
)


# ============================================================
# 10. PRECIPITATION-TYPE FREQUENCY MAPS
# ============================================================

freezing_rain_pct = (
    percentage_of_hours(
        freezing_rain
    )
)

freezing_rain_pct.name = (
    "freezing_rain_hours_pct"
)

freezing_rain_pct.attrs = {
    "long_name": (
        "Percentage of January hours "
        "classified as freezing rain"
    ),
    "units": "%",
}


freezing_drizzle_pct = (
    percentage_of_hours(
        freezing_drizzle
    )
)

freezing_drizzle_pct.name = (
    "freezing_drizzle_hours_pct"
)

freezing_drizzle_pct.attrs = {
    "long_name": (
        "Percentage of January hours "
        "classified as freezing drizzle"
    ),
    "units": "%",
}


wet_snow_pct = (
    percentage_of_hours(
        wet_snow
    )
)

wet_snow_pct.name = "wet_snow_hours_pct"

wet_snow_pct.attrs = {
    "long_name": (
        "Percentage of January hours "
        "classified as wet snow"
    ),
    "units": "%",
}


freezing_liquid_pct = (
    percentage_of_hours(
        freezing_liquid_precip
    )
)

freezing_liquid_pct.name = (
    "freezing_liquid_precip_hours_pct"
)

freezing_liquid_pct.attrs = {
    "long_name": (
        "Percentage of January hours classified "
        "as freezing rain or freezing drizzle"
    ),
    "units": "%",
}


accretion_relevant_pct = (
    percentage_of_hours(
        accretion_relevant_precip
    )
)

accretion_relevant_pct.name = (
    "accretion_relevant_precip_hours_pct"
)

accretion_relevant_pct.attrs = {
    "long_name": (
        "Percentage of January hours classified as "
        "freezing rain, freezing drizzle, or wet snow"
    ),
    "units": "%",
}


accretion_pct_of_precip_hours = (
    percentage_of_precip_hours(
        accretion_relevant_precip,
        precipitation_occurring,
    )
)

accretion_pct_of_precip_hours.name = (
    "accretion_relevant_pct_of_precip_hours"
)

accretion_pct_of_precip_hours.attrs = {
    "long_name": (
        "Percentage of precipitation hours classified "
        "as freezing rain, freezing drizzle, or wet snow"
    ),
    "units": "%",
}


# ============================================================
# 11. CONDITIONS DURING ACCRETION-RELEVANT PRECIPITATION
# ============================================================

# Instantaneous gust is temporally aligned with instantaneous
# ptype and is therefore the cleaner coincidence metric.
max_instant_gust_during_accretion = (
    era5["i10fg"]
    .where(accretion_relevant_precip)
    .max(
        dim="valid_time",
        skipna=True,
    )
)

max_instant_gust_during_accretion.name = (
    "max_instant_gust_during_accretion"
)

max_instant_gust_during_accretion.attrs = {
    "long_name": (
        "Maximum instantaneous 10 metre wind gust "
        "during freezing rain, freezing drizzle, or wet snow"
    ),
    "units": "m s-1",
}


# fg10 represents an interval maximum, so this associates
# the preceding-hour gust maximum with an event diagnosed
# at the corresponding valid timestamp.
max_interval_gust_during_accretion = (
    era5["fg10"]
    .where(accretion_relevant_precip)
    .max(
        dim="valid_time",
        skipna=True,
    )
)

max_interval_gust_during_accretion.name = (
    "max_interval_gust_during_accretion"
)

max_interval_gust_during_accretion.attrs = {
    "long_name": (
        "Maximum interval 10 metre wind gust associated "
        "with accretion-relevant precipitation timestamps"
    ),
    "units": "m s-1",
}


# avg_tprate is an interval-mean field while ptype is
# instantaneous. Treat this as an event-hour approximation,
# not exact instantaneous precipitation intensity.
mean_precip_rate_during_accretion = (
    era5["precip_rate_mmh"]
    .where(accretion_relevant_precip)
    .mean(
        dim="valid_time",
        skipna=True,
    )
)

mean_precip_rate_during_accretion.name = (
    "mean_precip_rate_during_accretion"
)

mean_precip_rate_during_accretion.attrs = {
    "long_name": (
        "Mean precipitation rate associated with "
        "freezing rain, freezing drizzle, or wet snow"
    ),
    "units": "mm h-1",
}


mean_tcslw_during_accretion = (
    era5["tcslw"]
    .where(accretion_relevant_precip)
    .mean(
        dim="valid_time",
        skipna=True,
    )
)

mean_tcslw_during_accretion.name = (
    "mean_tcslw_during_accretion"
)

mean_tcslw_during_accretion.attrs = {
    "long_name": (
        "Mean total-column supercooled liquid water "
        "during accretion-relevant precipitation"
    ),
    "units": era5["tcslw"].attrs.get(
        "units",
        "kg m-2",
    ),
}


# ============================================================
# 12. PRECIPITATION-TYPE DOMAIN SUMMARY
# ============================================================

ptype_values = ptype.values

valid_values = ptype_values[
    np.isfinite(ptype_values)
]

valid_values = valid_values[
    valid_values != 255
].astype(int)


unique_codes, counts = np.unique(
    valid_values,
    return_counts=True,
)


ptype_summary = pd.DataFrame({
    "code": unique_codes,
    "precipitation_type": [
        PTYPE_LABELS.get(
            int(code),
            "Unknown",
        )
        for code in unique_codes
    ],
    "grid_hour_count": counts,
})


ptype_summary[
    "pct_of_all_valid_grid_hours"
] = (
    ptype_summary["grid_hour_count"]
    / ptype_summary["grid_hour_count"].sum()
    * 100
)


ptype_summary.to_csv(
    OUTPUT_DIR / "ptype_domain_summary.csv",
    index=False,
)


print("\nPrecipitation types present:")
print(ptype_summary.to_string(index=False))


# ============================================================
# 13. ADD CRS TO COMBINED DATASET
# ============================================================

era5 = era5.rio.set_spatial_dims(
    x_dim="longitude",
    y_dim="latitude",
)

era5 = era5.rio.write_crs(
    "EPSG:4326"
)


# ============================================================
# 14. SAVE CLEANED COMBINED NETCDF
# ============================================================

encoding = {}

for var in era5.data_vars:

    if np.issubdtype(
        era5[var].dtype,
        np.number,
    ):

        encoding[var] = {
            "zlib": True,
            "complevel": 4,
        }


combined_path = (
    OUTPUT_DIR
    / "era5_icing_montana_2020_01.nc"
)


era5.to_netcdf(
    combined_path,
    encoding=encoding,
)


print(
    f"\nSaved combined NetCDF: "
    f"{combined_path}"
)


# ============================================================
# 15. QGIS-READY GEOTIFF OUTPUTS
# ============================================================

outputs = [

    # --------------------------------------------------------
    # General climate context
    # --------------------------------------------------------

    (
        mean_temp,
        "jan_2020_mean_temperature_c.tif",
    ),

    (
        freezing_pct,
        "jan_2020_freezing_hours_pct.tif",
    ),

    (
        mean_dewpoint_depression,
        "jan_2020_mean_dewpoint_depression_c.tif",
    ),

    (
        near_saturation_pct,
        "jan_2020_near_saturation_hours_pct.tif",
    ),

    (
        mean_low_cloud,
        "jan_2020_mean_low_cloud_cover_pct.tif",
    ),

    (
        low_cloud_50_pct,
        "jan_2020_low_cloud_ge50_hours_pct.tif",
    ),

    (
        median_cloud_base,
        "jan_2020_median_cloud_base_height_m.tif",
    ),


    # --------------------------------------------------------
    # Supercooled liquid water
    # --------------------------------------------------------

    (
        mean_tcslw,
        "jan_2020_mean_tcslw_kgm2.tif",
    ),

    (
        max_tcslw,
        "jan_2020_max_tcslw_kgm2.tif",
    ),

    (
        mean_tcslw_subfreezing,
        "jan_2020_mean_tcslw_subfreezing_kgm2.tif",
    ),


    # --------------------------------------------------------
    # Wind
    # --------------------------------------------------------

    (
        max_instant_gust,
        "jan_2020_max_instantaneous_gust_ms.tif",
    ),

    (
        p95_instant_gust,
        "jan_2020_p95_instantaneous_gust_ms.tif",
    ),

    (
        max_interval_gust,
        "jan_2020_max_interval_gust_ms.tif",
    ),

    (
        p95_interval_gust,
        "jan_2020_p95_interval_gust_ms.tif",
    ),


    # --------------------------------------------------------
    # Precipitation / snowfall
    # --------------------------------------------------------

    (
        mean_precip_rate,
        "jan_2020_mean_precip_rate_mmh.tif",
    ),

    (
        max_precip_rate,
        "jan_2020_max_precip_rate_mmh.tif",
    ),

    (
        mean_snowfall_rate,
        "jan_2020_mean_snowfall_rate_mmh.tif",
    ),

    (
        max_snowfall_rate,
        "jan_2020_max_snowfall_rate_mmh.tif",
    ),


    # --------------------------------------------------------
    # Precipitation-type analysis
    # --------------------------------------------------------

    (
        freezing_rain_pct,
        "jan_2020_pct_hours_freezing_rain.tif",
    ),

    (
        freezing_drizzle_pct,
        "jan_2020_pct_hours_freezing_drizzle.tif",
    ),

    (
        wet_snow_pct,
        "jan_2020_pct_hours_wet_snow.tif",
    ),

    (
        freezing_liquid_pct,
        "jan_2020_pct_hours_freezing_liquid_precip.tif",
    ),

    (
        accretion_relevant_pct,
        "jan_2020_pct_hours_accretion_relevant_precip.tif",
    ),

    (
        accretion_pct_of_precip_hours,
        "jan_2020_accretion_relevant_pct_of_precip_hours.tif",
    ),


    # --------------------------------------------------------
    # Conditions coinciding with accretion-relevant ptype
    # --------------------------------------------------------

    (
        max_instant_gust_during_accretion,
        "jan_2020_max_instant_gust_during_accretion_ms.tif",
    ),

    (
        max_interval_gust_during_accretion,
        "jan_2020_max_interval_gust_during_accretion_ms.tif",
    ),

    (
        mean_precip_rate_during_accretion,
        "jan_2020_mean_precip_rate_during_accretion_mmh.tif",
    ),

    (
        mean_tcslw_during_accretion,
        "jan_2020_mean_tcslw_during_accretion_kgm2.tif",
    ),
]


for da, filename in outputs:
    export_geotiff(
        da,
        filename,
    )


# ============================================================
# 16. DIAGNOSTICS
# ============================================================

print(
    "\n--- ERA5 ICING PIPELINE DIAGNOSTICS ---"
)

print(
    f"Time period: "
    f"{era5.valid_time.min().values} -> "
    f"{era5.valid_time.max().values}"
)

print(
    f"Grid: "
    f"{era5.sizes['latitude']} x "
    f"{era5.sizes['longitude']}"
)

print(
    f"Temperature range: "
    f"{float(era5['t2m_c'].min()):.2f} to "
    f"{float(era5['t2m_c'].max()):.2f} °C"
)

print(
    f"Max instantaneous gust: "
    f"{float(era5['i10fg'].max()):.2f} m/s"
)

print(
    f"Max interval gust: "
    f"{float(era5['fg10'].max()):.2f} m/s"
)

print(
    f"Max precipitation rate: "
    f"{float(era5['precip_rate_mmh'].max()):.3f} mm/h"
)

print(
    f"Max snowfall rate: "
    f"{float(era5['snowfall_rate_mmh'].max()):.3f} mm/h SWE"
)

print(
    f"Max total-column supercooled liquid water: "
    f"{float(era5['tcslw'].max()):.4f} "
    f"{era5['tcslw'].attrs.get('units', 'kg m-2')}"
)

print("\nProcessing complete.")

AVG variables:
['avg_tsrwe', 'avg_tprate']

INSTANT variables:
['d2m', 't2m', 'i10fg', 'cbh', 'lcc', 'tcslw']

MAX variables:
['fg10']

PTYPE variables:
['ptype']

Merged dimensions:
Frozen({'valid_time': 744, 'latitude': 21, 'longitude': 29})

Merged variables:
['d2m', 't2m', 'i10fg', 'cbh', 'lcc', 'tcslw', 'avg_tsrwe', 'avg_tprate', 'fg10', 'ptype']

Precipitation types present:
 code       precipitation_type  grid_hour_count  pct_of_all_valid_grid_hours
    0         No precipitation            89341                    19.717896
    1                     Rain            36037                     7.953502
    3            Freezing rain             1209                     0.266831
    5                     Snow           291037                    64.232966
    6                 Wet snow            23888                     5.272172
    7 Mixture of rain and snow             6694                     1.477391
    8              Ice pellets             4890                     1.079241
